<a href="https://colab.research.google.com/github/Nixtla/statsforecast/blob/readme_ets/examples/Forecasting_with_Auto_Arima_%26_ETS_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecasting with Auto Arima & Auto ETS.

In [ ]:
!pip install statsforecast

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import seasonal_naive, auto_arima, ses
from statsforecast.utils import AirPassengers

In [ ]:
# If you want to list all avaiaible models run this lines
#from statsforecast.models import __all__
#__all__

## Define horizon and split data accordingly

In [ ]:
horizon = 12
ap_train = AirPassengers[:-horizon]
ap_test = AirPassengers[-horizon:]


In [ ]:
# We define the train df. We use the index functionality to make the training a lot faster. 
series_train = pd.DataFrame(
    {
        'ds': pd.date_range(start='1949-01-01', periods=ap_train.size, freq='M'),
        'y': ap_train
    },
    index=pd.Index([0] * ap_train.size, name='unique_id')
)

In [ ]:
#Define the parameters that you want to use in your models. 
season_length=12 
# Note: For all models the following parameters are passed automaticly and don't need to be declared: (X, h, future_xreg)


fcst = StatsForecast(
    series_train, 
    models=[(auto_arima, season_length), (ets, season_lenght)], 
    freq='M', 
    n_jobs=1
)

# For some models like ARIMA , you can easly include confidence intervals
forecasts = fcst.forecast(horizon, level=(80, 95))

# forecasts = fcst.forecast(horizon) # Run this line if you don't want to include levels for ARIMA

In [ ]:

forecasts['y_test'] = ap_test


In [ ]:
#You analyse see the models that have been trained and the respective values by calling the DF.
forecasts

In [ ]:
#We are going to plot the models againts the real values of test.

fig, ax = plt.subplots(1, 1, figsize = (20, 7))
df_plot = pd.concat([series_train, forecasts]).set_index('ds')
df_plot[['y', 'y_test', 'auto_arima_season_length-12_mean','ets_season_lenght-12']].plot(ax=ax, linewidth=2)
ax.fill_between(df_plot.index, 
                df_plot['auto_arima_season_length-12_lo-80'], 
                df_plot['auto_arima_season_length-12_hi-80'],
                alpha=.35,
                color='green',
                label='auto_arima_level_80')
ax.fill_between(df_plot.index, 
                df_plot['auto_arima_season_length-12_lo-95'], 
                df_plot['auto_arima_season_length-12_hi-95'],
                alpha=.2,
                color='green',
                label='auto_arima_level_95')
ax.set_title('AirPassengers Forecast', fontsize=22)
ax.set_ylabel('Monthly Passengers', fontsize=20)
ax.set_xlabel('Timestamp [t]', fontsize=20)
ax.legend(prop={'size': 15})
ax.grid()
for label in (ax.get_xticklabels() + ax.get_yticklabels()):
    label.set_fontsize(20)

# Adding external regressors
Some models like auto_arima, have the support for external regresors or exogenous variables. We are working to expand this feature to other models.

In [ ]:
# First we are going to include new exogenous variables as columns to our train data_frame. (You can include things like weather or holidays)

series_train['trend'] = np.arange(1, ap_train.size + 1)
series_train['intercept'] = np.ones(ap_train.size)
series_train['month'] = series_train['ds'].dt.month
series_train = pd.get_dummies(series_train, columns=['month'], drop_first=True)

In [ ]:
# We consruct the test DF. We use the index functionality to make it faster.
xreg_test = pd.DataFrame(
    {
        'ds': pd.date_range(start='1960-01-01', periods=ap_test.size, freq='M')
    },
    index=pd.Index([0] * ap_test.size, name='unique_id')
)

In [ ]:
# We construct xreg for test. The train series ends at the 133th step. 

xreg_test['trend'] = np.arange(133, ap_test.size + 133)
xreg_test['intercept'] = np.ones(ap_test.size)
xreg_test['month'] = xreg_test['ds'].dt.month
xreg_test = pd.get_dummies(xreg_test, columns=['month'], drop_first=True)

In [ ]:
#Define the parameters that you want to use in your models. 
season_length=12

# Note: For all models the following parameters are passed automaticly and don't need to be declared: (X, h, future_xreg)

fcst = StatsForecast(
    series_train, 
    models=[(auto_arima, season_length), (seasonal_naive, season_length)], 
    freq='M', 
    n_jobs=1
)

# Include levels for auto_arima
forecasts = fcst.forecast(horizon, level=(80, 95))

In [ ]:
forecasts['y_test'] = ap_test